# App 6 · Anthropic Skills — 当 prompt 长到没法分享的时候

写过两年 LLM 应用的人都会遇到这个问题：你的核心提示词越来越长。一开始 50 字够，三个月后 500 字，半年后 2000 字带 5 个 few-shot 例子和 3 段边界条件说明。同事问你"能不能把这套提示词给我"，你发个 .txt 给他——但他粘进自己的项目还得调，因为你的 prompt 引用了你那边的工具、你的数据格式、你的命名约定。**prompt 不是孤立资产，它和它运行的上下文绑死**。

Anthropic 在 2024 年下半年开始推 [Skills](https://www.anthropic.com/news/agent-skills) 格式就是为了解决这件事。Skill 是一个**文件夹**，不是一个文件——里面有 prompt（`SKILL.md`）、有 helper 脚本（可选）、有按需加载的参考文档（也可选）。整个文件夹可以 git clone、可以放进 `~/.claude/skills/`、可以传到 [Claude.ai marketplace](https://claude.ai/skills)。**它把"prompt + 配套代码 + 文档"打包成一个原子单位**，这才是可分享、可版本化的能力资产。

```
my_skill/
├── SKILL.md           # 必需：YAML frontmatter (name + description) + body
├── helper.py          # 可选：Skill 可调用的脚本
└── reference/         # 可选：按需加载的文档
    └── checklist.md
```

Skills 还有一个杀手特性叫**Progressive Disclosure**——LLM 在不同时机加载不同深度的内容：

1. **启动**：只读所有 Skill 的 `description`（几十字一条），整体几百 token
2. **匹配**：用户来 query，LLM 看 description 决定调哪个 Skill，把那个 Skill 的 `body` 加载进来（几百字）
3. **深入**：执行过程中 Skill body 提到要查某个 reference/*.md，按需再加载

这么做的核心收益是 **省 context = 省钱 + 提速**。如果一个 LLM 客户端连了 50 个 Skills、Skill 又各有 10 页 reference，全量加载是 50 × 10 × 5000 = 250 万 token，每次推理都吃完——progressive disclosure 把它压到典型 query 只用 1-2 个 Skill 各几百 token。

这一节做三件事——理解 Skill 文件夹结构、用 LLM 路由 query 到合适 Skill、手写一个新 Skill 演示完整生命周期。还会看 Skills × MCP 集成模式：Skill 描述 workflow，MCP 提供 tool，两者配合。

> **跑这一节前**：跑过 [App5 MCP](./App5_MCP_Server.ipynb) 理解工具协议。本节 LLM 路由演示需要 `utils.config.setup()` 拿到可用 LLM 后端；离线时也能跑（传 mock LLM）。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"📂 repo root: {_root}")


📂 repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


## 1. Skills vs Prompts vs MCP

|  | Prompts | MCP | Skills |
|---|---|---|---|
| **解决** | 单次任务说明 | 外部能力接入 | **内化能力打包** |
| **形态** | 字符串/模板 | server 暴露 tool | 文件夹 (SKILL.md + scripts) |
| **生命周期** | 一次性 | 长期连接 | 按需载入 |
| **复用范围** | 单 prompt | 多 LLM 共享 | **跨项目跨团队** |
| **关键特性** | 灵活 | 跨厂可移植 | **Progressive Disclosure** |

### Skill 的样子

```
my_skill/
├── SKILL.md           # 必需: YAML frontmatter (name+description) + body
├── helper.py          # 可选: Claude 可调用的脚本
└── reference/         # 可选: 按需加载的文档
    └── checklist.md
```

### Progressive Disclosure（杀手特性）

1. **启动**: 只读 description（几十字）
2. **匹配**: query → 加载 body（几百字）
3. **细节**: 需要 → 加载 reference/*.md（按需）

→ **省 context = 省钱 + 提速**


In [2]:
from utils.skills_helpers import (
    parse_skill_md, validate_skill, discover_skills,
    match_skill_for_query, load_skill_progressive,
)
from utils.config import setup
env = setup()
llm = env.get_llm()

# Discover 已有的 3 个 example skills
skills = discover_skills("assets/enterprise_5days/skills_demo")
print(f"发现 {len(skills)} 个 example skills:")
for s in skills:
    print(f"  • {s.name} (v{s.version}): {s.description[:80]}...")


[OK] 已加载配置: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code\.env
课程环境配置:
  API Key:   ✓ 已配置
  LLM:       dashscope / qwen-plus
  Embedding: dashscope / text-embedding-v3


[LLM] dashscope / qwen-plus
发现 3 个 example skills:
  • enterprise-knowledge-assistant (v1.0): Answers employee questions about HR policy, technical APIs, product info, and ru...
  • code-review (v0.2): Performs a structured code review of Python changes — runs ruff for style, mypy ...
  • db-query (v0.1): Helps the user query the company's order / inventory database via the enterprise...


## 2. USE — 怎么用 Skill

学员视角：把 Skill 文件夹放到合适位置（`~/.claude/skills/` / git repo / Claude.ai marketplace），Claude 自动 discover + route。

下面演示：用户来一个 query → match_skill_for_query LLM 路由 → load body → 执行 workflow。


In [3]:
# Demo: 让 LLM 看 description 决定调哪个 skill
test_queries = [
    "review this Python function for security issues",
    "查 ORD-001 订单状态",
    "公司年假政策是什么",
    "写一首关于春天的诗",
]

for q in test_queries:
    picked = match_skill_for_query(q, skills, llm)
    name = picked.name if picked else "(无匹配 — 走默认 LLM)"
    print(f"  '{q[:36]}'  →  {name}")


  'review this Python function for secu'  →  code-review


  '查 ORD-001 订单状态'  →  db-query


  '公司年假政策是什么'  →  enterprise-knowledge-assistant


  '写一首关于春天的诗'  →  (无匹配 — 走默认 LLM)


In [4]:
# Progressive disclosure 演示：简单 query 不加载 reference，复杂 query 才加载
code_review_skill = next(s for s in skills if s.name == "code-review")

# 简单 query
loaded_simple = load_skill_progressive(code_review_skill, "review def add(a,b): return a+b", llm)
print(f"简单 query: 加载 reference {len(loaded_simple['references_loaded'])} 个，估算 tokens {loaded_simple['tokens_estimate']}")

# 复杂 query
loaded_full = load_skill_progressive(code_review_skill, "我团队对错误处理特别敏感，要走完整 checklist 严格审", llm)
print(f"复杂 query: 加载 reference {len(loaded_full['references_loaded'])} 个，估算 tokens {loaded_full['tokens_estimate']}")
print(f"\n💡 复杂 query 多读了 {loaded_full['tokens_estimate'] - loaded_simple['tokens_estimate']} tokens — 这就是 progressive disclosure")


简单 query: 加载 reference 0 个，估算 tokens 317


复杂 query: 加载 reference 1 个，估算 tokens 540

💡 复杂 query 多读了 223 tokens — 这就是 progressive disclosure


## 3. CREATE — 怎么创造 Skill

下面手写一个新 skill：`meeting_notes`。

### Step 1: 写 SKILL.md


In [5]:
import os, tempfile

skill_md = """---
name: meeting-notes
description: Helps the user summarize a meeting transcript or audio recording into structured notes — action items, decisions, follow-ups. Use when the user has a meeting recording or transcript and asks for a summary.
allowed-tools: [read_file]
version: "0.1"
---

# Meeting Notes Skill

## When to use

- "总结这场会议"
- "从录音里提取 action items"
- "这场会议有什么决定？"

## Workflow

1. **Identify input**: 文件路径 / 直接粘贴的 transcript
2. **Extract**:
   - **决定 (Decisions)**: 拍板的事项
   - **Action items**: 谁、做什么、何时
   - **Follow-ups**: 待跟进
3. **Format** as markdown
4. **Verify** with the user before sending out
"""

# 写到临时文件夹 + validate
with tempfile.TemporaryDirectory() as tmp:
    sd = os.path.join(tmp, "meeting-notes")
    os.makedirs(sd)
    with open(os.path.join(sd, "SKILL.md"), "w", encoding="utf-8") as f:
        f.write(skill_md)
    val = validate_skill(sd)
    fm, body = parse_skill_md(os.path.join(sd, "SKILL.md"))
    print(f"validate: ok={val['ok']}  warnings={val['warnings']}")
    print(f"name: {fm['name']}")
    print(f"description 长度: {len(fm['description'])} 字符")
    print(f"body 长度: {len(body)} 字符")


validate: ok=True  warnings=[]
name: meeting-notes
description 长度: 205 字符
body 长度: 326 字符


## 4. Skills × MCP 集成

Skill 描述「何时用 + 步骤」；MCP 提供「可调的 tool」。两者**配对**：

```
Skills              MCP
(能力)              (工具)
   ↓                   ↑
 教 Claude 何时用    暴露具体函数

例: db_query Skill → 调 enterprise-demo MCP server 的 query_order tool
```

`skills_demo/db_query/SKILL.md` 的 `allowed-tools` 字段限制它只能调 3 个 MCP tool：


In [6]:
# 看 db_query skill 的 frontmatter
db_skill = next((s for s in skills if s.name == "db-query"), None)
if db_skill:
    print(f"name: {db_skill.name}")
    print(f"allowed-tools: {db_skill.allowed_tools}")
    print(f"\n— body 前 300 字 —")
    print(db_skill.body[:300])


name: db-query
allowed-tools: ['mcp__enterprise-demo__query_order', 'mcp__enterprise-demo__check_inventory', 'mcp__enterprise-demo__send_notification']

— body 前 300 字 —
# DB Query Skill (via MCP)

A skill that demonstrates **Skills × MCP integration**. The skill body teaches
Claude how to translate natural language to the right MCP tool call; the actual
data access goes through the `enterprise-demo` MCP server.

## When to use

- "Look up order ORD-xxx"
- "How much


<!-- session-2026-04-29-superset-completion -->
## 4.5 实战：用 code-review Skill 审一段含 bug 的代码

前面 demo 了 skill 怎么发现 / 加载 / 匹配 / 集成 MCP。**但 skill 真正的工程价值是"把领域知识打包成可复用单元"——这里跑一次完整端到端流程**：

1. 给一段含 bug 的真实 Python 代码（多种安全漏洞）
2. 用 `match_skill_for_query` 路由到 `code-review` skill
3. 加载 skill body + 必要的 reference/checklist.md（progressive disclosure）
4. 把 skill 指令 + 待审代码喂给 LLM
5. 收集 LLM 审查结果

这是 Anthropic Skills 的标准工程化使用模式——所以 Claude Code / Claude Desktop 可以管理上百个 skill 而不爆 context。


In [ ]:
# 用 code-review Skill 审一段真实含 bug 的 Python 代码

BUGGY_CODE = """
import sqlite3

def get_user(username, password):
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    # BUG 1: SQL injection - direct string formatting in SQL
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    cursor.execute(query)
    user = cursor.fetchone()
    # BUG 2: connection never closed (resource leak)
    return user

def cache_data(key, data):
    cache = {}
    # BUG 3: cache is a local dict — new each call, useless cache
    cache[key] = data
    return cache[key]

def parse_json_response(response_text):
    import json
    # BUG 4: no try/except — malformed JSON raises uncaught exception
    return json.loads(response_text)["data"]
"""

# 1. Discovery + matching
print("=" * 78)
print("       Skill 端到端实战：用 code-review skill 审 buggy 代码")
print("=" * 78)
skills_dir = "../assets/enterprise_5days/skills_demo"  # main repo path to demo skills
import os
if not os.path.exists(skills_dir):
    skills_dir = "assets/enterprise_5days/skills_demo"  # if running from main repo root

try:
    skills = discover_skills(skills_dir)
    print(f"\n[1] Discovered {len(skills)} skills in {skills_dir}/")
    for s in skills:
        print(f"  - {s.name}: {s.description[:60]}...")
except Exception as e:
    print(f"\n[skip] discover_skills failed: {e}")
    skills = []

if skills:
    # 2. Route query to skill
    query = "请审查这段 Python 代码找出 bug 和安全问题"
    matched = match_skill_for_query(skills, query)
    print(f"\n[2] Query: '{query}'")
    print(f"    Matched skill: {matched.name if matched else 'none'}")

    if matched:
        # 3. Load skill body + relevant references (progressive disclosure)
        loaded = load_skill_progressive(matched, query=query)
        print(f"\n[3] Skill loaded:")
        print(f"    body: {len(loaded['body'])} chars")
        print(f"    references_loaded: {[r['name'] for r in loaded.get('references_loaded', [])]}")

        # 4. Build LLM prompt = skill instruction + buggy code
        full_prompt = f"""{loaded['body']}

待审查的代码：
```python
{BUGGY_CODE}
```

请按 SKILL.md 中的指引输出审查报告。
"""
        print(f"\n[4] 完整 prompt 长度: {len(full_prompt)} chars")
        print(f"    （生产里这一步是真实 LLM 调用：调 Ollama / OpenAI / Claude）")
        print(f"\n[5] 模拟 LLM 审查输出（基于 skill checklist + 代码扫描）：")
        print("-" * 78)
        # 简化版：这里 enumerate 已知的 bug，生产里是 LLM 输出
        print("""
找到 4 个问题：

[CRITICAL] SQL Injection (line 6):
  query = f"SELECT * FROM users WHERE username = '{username}'..."
  → 应使用参数化查询：cursor.execute(query, (username, password))

[HIGH] Resource Leak (line 9):
  conn 从未关闭，长期运行会耗尽 DB 连接池
  → 用 with sqlite3.connect(...) as conn: 上下文管理器

[MEDIUM] 无效缓存 (line 14):
  cache = {} 在函数内每次调用都新建，缓存无效
  → 用模块级 dict 或 functools.lru_cache 装饰器

[MEDIUM] 未处理异常 (line 21):
  json.loads 对畸形输入会抛 JSONDecodeError，未捕获
  → 用 try/except + 默认返回 None 或 raise 业务异常
""")
        print("-" * 78)
        print("\n这就是 Skills 的工程价值：")
        print("  - SKILL.md 一次写好，多个项目复用")
        print("  - 关键的"如何审 SQL injection"等知识沉淀在 skill 里")
        print("  - LLM 自动按 skill 指引执行，不需要每次重复 prompt")
else:
    print("\n(skills_demo 不可用，跳过实战 demo)")

print("=" * 78)


In [ ]:
# 自检：会用 + 会写 Skill 了吗？
def verify_app6() -> bool:
    print("=" * 56)
    print("自检 · App6 Anthropic Skills")
    print("=" * 56)
    checks: list[tuple[str, bool, str]] = []

    # 1. discover ≥3 skills
    try:
        n_skills = len(skills)  # noqa: F821 —— §1 定义
        checks.append(("discover_skills 找到 ≥3 个 skill", n_skills >= 3, f"{n_skills} 个"))
    except (NameError, TypeError):
        checks.append(("discover_skills 找到 ≥3 个 skill", False, "⏭ 跳过 §1"))

    # 2. 必填字段齐全
    try:
        all_have_required = all(s.name and s.description for s in skills)  # noqa: F821
        checks.append(("每个 skill 必填字段齐全 (name+description)",
                       all_have_required, "ok" if all_have_required else "有 skill 缺字段"))
    except (NameError, AttributeError, TypeError):
        checks.append(("每个 skill 必填字段齐全", False, "⏭ skills 不存在或类型异常"))

    # 3. Progressive disclosure：复杂 query 多读 reference
    try:
        delta = loaded_full["tokens_estimate"] - loaded_simple["tokens_estimate"]  # noqa: F821
        ok = delta > 0
        checks.append(("Progressive disclosure 生效（复杂 query 多读 reference）",
                       ok, f"差 {delta} tokens"))
    except (NameError, KeyError, TypeError):
        checks.append(("Progressive disclosure 生效", False, "⏭ §2 progressive 演示未跑"))

    # 4. db-query Skill 配了 allowed-tools（Skills × MCP 集成证据）
    try:
        n_at = len(db_skill.allowed_tools) if db_skill else 0  # noqa: F821
        checks.append(("db-query Skill 配置了 allowed-tools", n_at > 0, f"{n_at} 个 MCP tool"))
    except (NameError, AttributeError):
        checks.append(("db-query Skill 配置了 allowed-tools", False, "⏭ §4 cell 未跑"))

    passed = sum(1 for _, ok, _ in checks if ok)
    for name, ok, detail in checks:
        icon = "✅" if ok else ("⏭" if detail.startswith("⏭") else "❌")
        print(f"  {icon} {name}  ({detail})")
    print(f"\n通过 {passed}/{len(checks)}")
    if passed == len(checks):
        print("下一节：App7_LLMOps")
    elif passed >= 2:
        print("部分通过——把跳过的 cell 跑完后重跑这一格。")
    else:
        print("未通过——回到顶部按顺序 Run All。")
    return passed == len(checks)


verify_app6()


## 5. 总结

- **Skills = 可打包内化能力**：跨项目跨团队复用
- **三层载入** (progressive disclosure)：name+description always；body 匹配后；reference 按需 → 省 context
- **配 MCP**：Skill 描述 workflow，MCP 提供 tool，两者解耦
- **生产**: 团队共享 git repo / 个人 ~/.claude/skills / Claude.ai marketplace

**下一步**:
- App7_LLMOps — observability + trace + cost
- 5 天版 Day 4 下午: 完整【基础+进阶+verify】练习（写 meeting-notes / score description / Skills × MCP 集成）


<!-- session-2026-04-29-teaching-pass -->
---

## 参考实现：三个完整 Skill 范例

本 notebook 解释了 Anthropic Skills 的格式与 progressive disclosure 机制。课程仓里有三个开箱即用的 Skill 范例可以直接 fork 改造：

**[`assets/enterprise_5days/skills_demo/`](../assets/enterprise_5days/skills_demo/)**

| 范例 | 用途 | 复杂度 |
|------|------|--------|
| [`code_review/`](../assets/enterprise_5days/skills_demo/code_review/) | 代码审查 skill：YAML frontmatter + body + reference/checklist.md，最小可用形态 | ⭐ |
| [`db_query/`](../assets/enterprise_5days/skills_demo/db_query/) | DB 查询 skill：演示 **Skill × MCP 集成**——skill body 里调本地 MCP server 跑 SQL | ⭐⭐ |
| [`capstone_assistant/`](../assets/enterprise_5days/skills_demo/capstone_assistant/) | 企业级 skill：把整个升级 Capstone（Multi-Agent + RAG + 评测）打包成一个可分发的 skill | ⭐⭐⭐ |

每个范例都包含：
- `SKILL.md` — frontmatter (name + description) + body 指令
- `helper.py` 或 `pipeline.py` — 可被 skill 调用的 Python 函数
- `reference/` — 按需加载的细节文档（progressive disclosure）

**怎么用：**
1. 把整个 skill 文件夹复制到你的 `~/.claude/skills/` 或项目 `.claude/skills/` 下
2. Claude Code / Claude.ai 会自动 discover frontmatter
3. 当用户 query 命中 description，Claude 加载 body；只在需要时再读 reference/

工程化要点见 `assets/enterprise_5days/instructor/Day4_下午_MCP与Skills.ipynb` 的 Skills 章节。
